# Heatmap Defense Improvements — Ablations (ideas 1–4, 7)

Improves the frozen **Attn-last** CAM-intersection baseline from
`../attention_defense/` without editing that folder.

| ID | Variant | Change |
|----|---------|--------|
| 0 | `attn_last_baseline` | Intersection + mean-fill (control) |
| 1a | `gated_peakiness` | Skip mask when saliency looks clean-like |
| 1b | `gated_disagree` | Skip mask when EN/ZH predictions agree |
| 2 | `union_masks` | Per-model thr then OR |
| 3 | `blur_fill` | Blur inside mask |
| 4a | `cc_filter` | Keep top-2 connected components |
| 4b | `cc_bbox` | Top-2 components + bounding-box snap |
| 7 | `peaked_heads` | Average lowest-entropy attention heads |

Protocol: tune on 100 → full 1000 multilingual attack. Cost stays ~4 passes/image.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets', 'matplotlib',
                'Pillow', 'scipy'], check=False)


In [ ]:
import os, platform, random, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor
from scipy import ndimage

os.makedirs('results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
BLUR_RADIUS  = 12
THRESHOLDS   = [0.75, 0.80, 0.85, 0.90, 0.95]
PEAKINESS_GATES = [0.05, 0.08, 0.10, 0.12, 0.15]  # max coverage to still skip mask

CLASSES = {
    'en': ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck'],
    'zh': ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}
LANGS = ['en', 'zh']


## 1. Models


In [ ]:
def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    arch = 'ViT-B-32'
    def __init__(self, arch='ViT-B-32'):
        self.arch = arch
        self.m, _, self.pp = open_clip.create_model_and_transforms(arch, pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer(arch)
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained(
            'OFA-Sys/chinese-clip-vit-base-patch16',
            attn_implementation='eager').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True,
                   return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'],
                                       attention_mask=t['attention_mask'],
                                       token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

def classify_batch(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

models = {}
t0 = time.time(); print('Loading en...', end=' ', flush=True)
models['en'] = EnCLIP('ViT-B-32'); print(f'{time.time()-t0:.1f}s')
t0 = time.time(); print('Loading zh...', end=' ', flush=True)
models['zh'] = ZhCLIP(); print(f'{time.time()-t0:.1f}s')
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}
print('Models ready.')


## 2. Dataset + multilingual attack


In [ ]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

all_idx  = np.arange(len(clean_224))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean_224)} images; tune subset = {len(tune_idx)}')

_FONT_CACHE = {}
def _font_paths():
    if platform.system() == 'Windows':
        wf = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        return (os.path.join(wf, 'msyh.ttc'), os.path.join(wf, 'arial.ttf'))
    return None, None
_CJK_FONT, _LAT_FONT = _font_paths()

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2]<=b[0] or b[2]<=a[0] or a[3]<=b[1] or b[3]<=a[1])

def _random_nonoverlapping_rect(rng_, bw, bh, placed):
    x_hi = max(0, DISPLAY_SIZE-bw); y_hi = max(0, DISPLAY_SIZE-bh)
    rx = ry = 0
    for _ in range(64):
        rx = rng_.randint(0, x_hi) if x_hi > 0 else 0
        ry = rng_.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rx, ry, rx+bw, ry+bh)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rx, ry, rx+bw, ry+bh)

def draw_multilingual_attack(img, en_word, zh_word, img_idx, already_224=False):
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    placed = []
    for box_i, (word, fp) in enumerate([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)]):
        font = _get_font(fp)
        bb   = draw.textbbox((0,0), word, font=font)
        bw   = (bb[2]-bb[0]) + 2*PAD
        bh   = (bb[3]-bb[1]) + PAD + 12
        rng_ = random.Random(int(img_idx)*NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng_, bw, bh, placed)
        placed.append(rect)
        rx, ry, rx2, ry2 = rect
        draw.rectangle([rx, ry, rx2, ry2], fill='white')
        draw.text((rx+PAD-bb[0], ry+PAD-bb[1]), word, fill='black', font=font)
    return img

attacked_imgs = [
    draw_multilingual_attack(clean_224[i], CLASSES['en'][target[i]],
                              CLASSES['zh'][target[i]], i, already_224=True)
    for i in range(len(clean_224))
]

clean_acc      = {ml: float((classify_batch(models[ml], clean_224, CLASSES[ml]) == true).mean()) for ml in LANGS}
preds_attacked = {ml: classify_batch(models[ml], attacked_imgs, CLASSES[ml]) for ml in LANGS}
baseline_acc   = {ml: float((preds_attacked[ml] == true).mean())   for ml in LANGS}
baseline_asr   = {ml: float((preds_attacked[ml] == target).mean()) for ml in LANGS}
print('Clean acc:   ', {k: f'{100*v:.1f}%' for k,v in clean_acc.items()})
print('Attacked acc:', {k: f'{100*v:.1f}%' for k,v in baseline_acc.items()})
print('Attacked ASR:', {k: f'{100*v:.1f}%' for k,v in baseline_asr.items()})


## 3. Attn-last saliency (+ peaked-heads option)


In [ ]:
def _norm_cam(cam):
    cam = np.maximum(cam if isinstance(cam, np.ndarray) else cam.cpu().numpy(), 0)
    cam -= cam.min()
    mx   = cam.max()
    return cam / mx if mx > 0 else cam

def align_cam(cam, size=DISPLAY_SIZE):
    return np.array(
        Image.fromarray((cam*255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    ) / 255.0

def _make_en_hook(collector):
    def hook(module, inputs, output):
        q_in = inputs[0]
        if getattr(module, 'batch_first', False):
            B, L, D = q_in.shape
        else:
            L, B, D = q_in.shape
            q_in = q_in.transpose(0, 1).contiguous()
        n_head = module.num_heads
        hd     = D // n_head
        with torch.no_grad():
            qkv = F.linear(q_in, module.in_proj_weight, module.in_proj_bias)
            q, k, _ = qkv.chunk(3, dim=-1)
            q = q.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            k = k.reshape(B, L, n_head, hd).permute(0, 2, 1, 3)
            attn = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
            attn = attn.softmax(dim=-1)
        collector.append(attn[0].detach().cpu())
    return hook

def _build_attn_cam(all_attns, variant='last', head_mode='mean', peaked_k=4):
    """Turn collected per-layer attention into a spatial cam.
    head_mode: 'mean' averages all heads; 'peaked' keeps the peaked_k lowest-entropy heads.
    """
    a = all_attns[-1]  # (nh, L, L) for last-layer
    if head_mode == 'peaked':
        # entropy of CLS→patch distribution per head
        cls = a[:, 0, 1:].clamp(min=1e-8)
        cls = cls / cls.sum(-1, keepdim=True)
        ent = -(cls * cls.log()).sum(-1)  # (nh,)
        keep = torch.argsort(ent)[:min(peaked_k, ent.numel())]
        cls_row = a[keep].mean(0)[0, 1:]
    else:
        cls_row = a.mean(0)[0, 1:]
    if variant == 'rollout':
        L = all_attns[0].shape[-1]
        rollout = torch.eye(L)
        for att in all_attns:
            a_r = 0.5 * att.mean(0) + 0.5 * torch.eye(L)
            a_r = a_r / a_r.sum(-1, keepdim=True)
            rollout = a_r @ rollout
        cls_row = rollout[0, 1:]
    n = int(round(cls_row.shape[0] ** 0.5))
    return _norm_cam(cls_row.reshape(n, n).numpy())

def classify_and_attn_en(pil_img, variant='last', head_mode='mean', peaked_k=4):
    wrapper   = models['en']
    x         = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    collector = []
    handles   = [rb.attn.register_forward_hook(_make_en_hook(collector))
                 for rb in wrapper.m.visual.transformer.resblocks]
    with torch.no_grad():
        feat = wrapper.m.visual(x)
        imf  = F.normalize(feat, dim=-1)
        pred = int((imf @ TEXT_EMB['en'].T).squeeze().argmax().item())
    for h in handles: h.remove()
    return pred, _build_attn_cam(collector, variant, head_mode, peaked_k)

def classify_and_attn_zh(pil_img, variant='last', head_mode='mean', peaked_k=4):
    wrapper = models['zh']
    pv = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        vis_out  = wrapper.m.vision_model(pixel_values=pv, output_attentions=True)
        proj_out = wrapper.m.visual_projection(vis_out.pooler_output)
        imf      = F.normalize(proj_out, dim=-1)
        pred     = int((imf @ TEXT_EMB['zh'].T).squeeze().argmax().item())
    attns = [a[0].cpu() for a in vis_out.attentions]
    return pred, _build_attn_cam(attns, variant, head_mode, peaked_k)

print('Saliency ready.')


## 4. Masking helpers (union / blur / CC / gate)


In [ ]:
def n_cam_intersection(*cams):
    return np.minimum.reduce([align_cam(c) for c in cams])

def dilate_mask(mask, iterations=3):
    m = mask.astype(bool)
    for _ in range(iterations):
        pad = np.pad(m, 1, mode='constant', constant_values=False)
        m = (pad[:-2,:-2]|pad[:-2,1:-1]|pad[:-2,2:]|
             pad[1:-1,:-2]|pad[1:-1,1:-1]|pad[1:-1,2:]|
             pad[2:,:-2]  |pad[2:,1:-1]  |pad[2:,2:])
    return m

def cam_to_mask(saliency, threshold=0.85, dilate=3):
    thr  = np.percentile(saliency, threshold * 100)
    mask = saliency >= thr
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def cam_to_mask_union(cam_en, cam_zh, thr_en=0.95, thr_zh=0.95, dilate=3):
    """Threshold each cam separately, then OR."""
    m_en = cam_to_mask(align_cam(cam_en), thr_en, dilate=0)
    m_zh = cam_to_mask(align_cam(cam_zh), thr_zh, dilate=0)
    mask = m_en | m_zh
    if dilate > 0: mask = dilate_mask(mask, iterations=dilate)
    return mask

def filter_mask_components(mask, top_k=2, bbox_snap=False):
    """Keep largest top_k connected components; optional AABB snap."""
    labeled, n = ndimage.label(mask.astype(bool))
    if n == 0:
        return mask.astype(bool)
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    keep = set(np.argsort(sizes)[::-1][:top_k] + 1)
    out = np.zeros_like(mask, dtype=bool)
    for i in keep:
        comp = labeled == i
        if bbox_snap:
            ys, xs = np.where(comp)
            out[ys.min():ys.max()+1, xs.min():xs.max()+1] = True
        else:
            out |= comp
    return out

def apply_mask(pil_img, mask, fill='mean'):
    arr = np.array(pil_img.convert('RGB'))
    m   = mask.astype(bool)
    if mask.shape != arr.shape[:2]:
        m = np.array(Image.fromarray(m.astype(np.uint8)*255).resize(
                     arr.shape[1::-1], Image.NEAREST)) > 127
    out = arr.copy()
    if fill == 'blur':
        blurred = np.array(Image.fromarray(arr).filter(
            ImageFilter.GaussianBlur(radius=BLUR_RADIUS)))
        out[m] = blurred[m]
    else:
        mean = arr[~m].mean(0) if (~m).any() else arr.reshape(-1, 3).mean(0)
        out[m] = mean
    return Image.fromarray(out.astype(np.uint8))

def saliency_peakiness(saliency):
    """Fraction of pixels above 95th percentile of the aligned map (lower = peakier)."""
    s = align_cam(saliency)
    thr = np.percentile(s, 95)
    return float((s >= thr).mean())

def should_defend(cam_en, cam_zh, pred_en, pred_zh, mode='peakiness',
                  peakiness_max=0.10):
    """Return True if we should apply the mask.
    mode='peakiness': defend when intersection coverage-at-p95 is ABOVE peakiness_max
                      (spread attention → clean-like → skip; tight spike → defend).
    Actually: peakiness metric as defined is coverage of top 5% — for a sharp spike
    on a small sticker the mass is concentrated so AFTER percentile threshold the
    binary mask is small. We use intersection mean above a high thr as 'hot coverage'.
    """
    if mode == 'disagree':
        return pred_en != pred_zh
    # peakiness: use coverage of pixels above 0.95 of the intersection map
    inter = n_cam_intersection(cam_en, cam_zh)
    cov = float((inter >= np.percentile(inter, 95)).mean())
    # Attacked images have TINY hot regions → low cov. Clean have large → high cov.
    # Defend when cov is LOW (spike). Skip when cov is HIGH (spread).
    return cov <= peakiness_max

print('Masking helpers ready.')


## 5. Variant runner


In [ ]:
def extract_pair(img, head_mode='mean', peaked_k=4):
    pred_en, cam_en = classify_and_attn_en(img, 'last', head_mode, peaked_k)
    pred_zh, cam_zh = classify_and_attn_zh(img, 'last', head_mode, peaked_k)
    return pred_en, pred_zh, cam_en, cam_zh

def build_mask(cam_en, cam_zh, cfg):
    mode = cfg.get('mask_mode', 'intersection')
    thr  = cfg.get('threshold', 0.95)
    dil  = cfg.get('dilate', 3)
    if mode == 'union':
        mask = cam_to_mask_union(cam_en, cam_zh, thr, thr, dilate=dil)
    else:
        inter = n_cam_intersection(cam_en, cam_zh)
        mask  = cam_to_mask(inter, thr, dilate=dil)
    if cfg.get('cc_filter'):
        mask = filter_mask_components(mask, top_k=cfg.get('top_k', 2),
                                      bbox_snap=cfg.get('bbox_snap', False))
    return mask

def run_variant(name, cfg, indices, images, label=''):
    """Run one ablation variant. Returns metrics dict for these indices."""
    imgs_sub = [images[i] for i in indices]
    true_sub = true[indices]
    tgt_sub  = target[indices]
    n = len(indices)
    head_mode = cfg.get('head_mode', 'mean')
    peaked_k  = cfg.get('peaked_k', 4)
    fill      = cfg.get('fill', 'mean')
    gate      = cfg.get('gate')  # None | 'peakiness' | 'disagree'
    peak_max  = cfg.get('peakiness_max', 0.10)

    print(f'  [{name}] {label} n={n}...', end=' ', flush=True)
    t0 = time.time()

    masked_imgs = []
    coverages = []
    gated_off = 0
    for img in imgs_sub:
        pred_en, pred_zh, cam_en, cam_zh = extract_pair(img, head_mode, peaked_k)
        do_mask = True
        if gate is not None:
            do_mask = should_defend(cam_en, cam_zh, pred_en, pred_zh,
                                    mode=gate, peakiness_max=peak_max)
            if not do_mask:
                gated_off += 1
                masked_imgs.append(img)
                coverages.append(0.0)
                continue
        mask = build_mask(cam_en, cam_zh, cfg)
        coverages.append(float(mask.mean()))
        masked_imgs.append(apply_mask(img, mask, fill=fill))

    new_preds = {ml: classify_batch(models[ml], masked_imgs, CLASSES[ml]) for ml in LANGS}
    elapsed = time.time() - t0
    print(f'{elapsed:.1f}s  gated_off={gated_off}/{n}  cov={100*np.mean(coverages):.1f}%')

    return {
        'preds': new_preds,
        'true': true_sub,
        'target': tgt_sub,
        'coverage': float(np.mean(coverages)),
        'gated_off_frac': gated_off / n,
        'time_s': elapsed,
    }

# Variant configs
VARIANTS = {
    'attn_last_baseline': dict(mask_mode='intersection', fill='mean', threshold=0.95),
    'gated_peakiness':    dict(mask_mode='intersection', fill='mean', threshold=0.95,
                               gate='peakiness', peakiness_max=0.10),
    'gated_disagree':     dict(mask_mode='intersection', fill='mean', threshold=0.95,
                               gate='disagree'),
    'union_masks':        dict(mask_mode='union', fill='mean', threshold=0.95),
    'blur_fill':          dict(mask_mode='intersection', fill='blur', threshold=0.95),
    'cc_filter':          dict(mask_mode='intersection', fill='mean', threshold=0.95,
                               cc_filter=True, top_k=2, bbox_snap=False),
    'cc_bbox':            dict(mask_mode='intersection', fill='mean', threshold=0.95,
                               cc_filter=True, top_k=2, bbox_snap=True),
    'peaked_heads':       dict(mask_mode='intersection', fill='mean', threshold=0.95,
                               head_mode='peaked', peaked_k=4),
}

print('Variants:', list(VARIANTS))


## 6. Tune on 100


In [ ]:
# Tune threshold (and peakiness gate) on 100-image subset for each variant
best_cfg = {}
tune_imgs = [attacked_imgs[i] for i in tune_idx]

for name, base_cfg in VARIANTS.items():
    print(f'\n=== Tuning {name} ===')
    best_acc, best = -1.0, dict(base_cfg)
    thr_grid = THRESHOLDS
    peak_grid = PEAKINESS_GATES if base_cfg.get('gate') == 'peakiness' else [base_cfg.get('peakiness_max', 0.10)]

    for thr in thr_grid:
        for pk in peak_grid:
            cfg = dict(base_cfg, threshold=thr, peakiness_max=pk)
            res = run_variant(name, cfg, tune_idx, attacked_imgs, f'thr={thr}')
            en_acc = float((res['preds']['en'] == res['true']).mean())
            if en_acc > best_acc:
                best_acc = en_acc
                best = cfg
                best['_tune_en_acc'] = en_acc
                best['_tune_cov'] = res['coverage']
    best_cfg[name] = best
    print(f'  BEST thr={best["threshold"]} peak={best.get("peakiness_max")} '
          f'EN={100*best["_tune_en_acc"]:.1f}% cov={100*best["_tune_cov"]:.1f}%')

with open('results/tune_best_cfg.json', 'w', encoding='utf-8') as f:
    # strip non-serializable
    dump = {k: {kk: vv for kk, vv in v.items() if not callable(vv)} for k, v in best_cfg.items()}
    json.dump(dump, f, indent=2)
print('Saved results/tune_best_cfg.json')


## 7. Full 1000 + clean degradation


In [ ]:
# Full 1000 eval + clean degradation for each tuned variant
full_results = {}

for name, cfg in best_cfg.items():
    print(f'\n=== Full eval {name} ===')
    atk = run_variant(name, cfg, all_idx, attacked_imgs, 'attacked')
    cln = run_variant(name, cfg, all_idx, clean_224, 'clean')

    defense = {}
    for ml in LANGS:
        defense[ml] = {
            'acc': float((atk['preds'][ml] == true).mean()),
            'asr': float((atk['preds'][ml] == target).mean()),
            'baseline_acc': baseline_acc[ml],
            'baseline_asr': baseline_asr[ml],
        }
    clean_deg = {
        ml: {
            'baseline_acc': clean_acc[ml],
            'masked_acc': float((cln['preds'][ml] == true).mean()),
            'delta_acc': float((cln['preds'][ml] == true).mean()) - clean_acc[ml],
        } for ml in LANGS
    }
    row = {
        'variant': name,
        'config': {k: v for k, v in cfg.items() if not str(k).startswith('_')},
        'n_images': int(len(all_idx)),
        'inference_cost': 4,  # same as Attn-last (2 fwd saliency + 2 reclass); gate skips reclass sometimes
        'coverage_mean': atk['coverage'],
        'gated_off_frac': atk['gated_off_frac'],
        'defense': defense,
        'clean_degradation': clean_deg,
        'defense_acc_mean': float(np.mean([defense[ml]['acc'] for ml in LANGS])),
        'defense_asr_mean': float(np.mean([defense[ml]['asr'] for ml in LANGS])),
        'clean_delta_mean': float(np.mean([clean_deg[ml]['delta_acc'] for ml in LANGS])),
    }
    out = f'results/confusion_results_{name}.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(row, f, indent=2, ensure_ascii=False)
    full_results[name] = row
    print(f'  mean acc={100*row["defense_acc_mean"]:.1f}%  '
          f'clean Δ={100*row["clean_delta_mean"]:+.1f}pp  → {out}')

# Summary table + chart
print('\n' + '='*90)
print(f'  {"Variant":<22} {"Mean":>7} {"EN":>7} {"ZH":>7} {"Cov":>6} {"CleanΔ":>8}')
print('='*90)
for name, r in full_results.items():
    print(f'  {name:<22} '
          f'{100*r["defense_acc_mean"]:>6.1f}% '
          f'{100*r["defense"]["en"]["acc"]:>6.1f}% '
          f'{100*r["defense"]["zh"]["acc"]:>6.1f}% '
          f'{100*r["coverage_mean"]:>5.1f}% '
          f'{100*r["clean_delta_mean"]:>+7.1f}pp')
print('='*90)

names = list(full_results)
x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(max(8, 1.2*len(names)), 4.5))
ax.bar(x - w/2, [100*full_results[n]['defense_acc_mean'] for n in names], w, label='Attacked mean acc')
ax.bar(x + w/2, [100*full_results[n]['clean_delta_mean'] for n in names], w, label='Clean Δ (pp)')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right', fontsize=8)
ax.axhline(72.6, color='gray', ls=':', lw=1, label='published Attn-last 72.6%')
ax.set_ylabel('%')
ax.set_title('Heatmap defense improvements — multilingual n=1000')
ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/final_comparison.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved results/final_comparison.png')

with open('results/comparison_summary.json', 'w', encoding='utf-8') as f:
    json.dump(full_results, f, indent=2, ensure_ascii=False)
print('Saved results/comparison_summary.json')
